03_stage1_models

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV, cross_validate, cross_val_predict
from sklearn.metrics import (make_scorer, precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, accuracy_score, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.inspection import permutation_importance

SEED = 42
FAST = False          # True = smaller grids and fewer trees, for a quick trial run
OUT_DIR = Path("outputs")

stage1 = pd.read_pickle(OUT_DIR / "stage1_all_women.pkl")
fl = pd.read_pickle(OUT_DIR / "feature_lists.pkl")
CAT, NUM = list(fl["FEATURES_CAT"]), list(fl["FEATURES_NUM"])
FEATURES = CAT + NUM
VL = pd.read_pickle(OUT_DIR / "value_labels.pkl")
SRC = {"sector": "b1q3_perv1", "social_group": "b3q4_hhv1", "religion": "b3q3_hhv1",
       "state": "state_perv1", "marital": "b4q7_perv1"}

def lab(series, var):
    m = {}
    for k, v in VL.get(SRC.get(var, var), {}).items():
        try: m[str(int(float(k)))] = v
        except Exception: m[str(k)] = v
    s = series.astype(str).str.replace(r"\.0$", "", regex=True)
    return s.map(m).fillna(s)

y, groups = stage1["lfp"], stage1["HHID"]

# Final test set: ~20% of households, used once at the end
train_idx, test_idx = next(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
                           .split(stage1, y, groups))
tr, te = stage1.iloc[train_idx], stage1.iloc[test_idx]
Xtr, ytr, gtr = tr[FEATURES], tr["lfp"], tr["HHID"]
Xte, yte = te[FEATURES], te["lfp"]
assert set(tr["HHID"]).isdisjoint(set(te["HHID"])), "household leakage between train and test"

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)   # inner CV on train only
print("Train:", len(tr), "| Test:", len(te), "| rates: %.3f / %.3f" % (ytr.mean(), yte.mean()))

In [ ]:
def make_pipe(model):
    pre = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30), CAT),
        ("num", StandardScaler(), NUM)])
    return Pipeline([("pre", pre), ("model", model)])

# Decision tree: tune depth and minimum leaf size on PR-AUC
dt_grid = {"model__max_depth": [4, 8] if FAST else [4, 6, 8, 12],
           "model__min_samples_leaf": [50] if FAST else [20, 50, 100]}
gs_dt = GridSearchCV(make_pipe(DecisionTreeClassifier(class_weight="balanced", random_state=SEED)),
                     dt_grid, scoring="average_precision", cv=cv, n_jobs=-1)
gs_dt.fit(Xtr, ytr, groups=gtr)
print("Best tree:", gs_dt.best_params_, "| CV PR-AUC %.3f" % gs_dt.best_score_)

# Random forest: tune depth and minimum leaf size
rf_grid = {"model__max_depth": [10] if FAST else [10, 16, None],
           "model__min_samples_leaf": [30] if FAST else [10, 30]}
rf_base = RandomForestClassifier(n_estimators=60 if FAST else 200, max_features="sqrt",
                                 class_weight="balanced_subsample", n_jobs=-1, random_state=SEED)
gs_rf = GridSearchCV(make_pipe(rf_base), rf_grid, scoring="average_precision", cv=cv, n_jobs=1)
gs_rf.fit(Xtr, ytr, groups=gtr)
print("Best forest:", gs_rf.best_params_, "| CV PR-AUC %.3f" % gs_rf.best_score_)

In [ ]:
models = {
    "Majority baseline":   make_pipe(DummyClassifier(strategy="most_frequent")),
    "Logistic regression": make_pipe(LogisticRegression(max_iter=1000, class_weight="balanced")),
    "Decision tree":       gs_dt.best_estimator_,
    "Random forest":       gs_rf.best_estimator_,
}
scoring = {"accuracy": "accuracy", "precision": make_scorer(precision_score, zero_division=0),
           "recall": "recall", "f1": "f1", "roc_auc": "roc_auc", "pr_auc": "average_precision"}

rows = []
for name, pipe in models.items():
    res = cross_validate(clone(pipe), Xtr, ytr, groups=gtr, cv=cv, scoring=scoring)
    rows.append({"model": name, **{m: res[f"test_{m}"].mean() for m in scoring},
                 "pr_auc_sd": res["test_pr_auc"].std()})
cv_results = pd.DataFrame(rows).round(3)
display(cv_results)
cv_results.to_csv(OUT_DIR / "stage1_cv_results.csv", index=False)

cands = cv_results[cv_results["model"] != "Majority baseline"].sort_values(["pr_auc", "f1"], ascending=False)
BEST = cands.iloc[0]["model"]
print("Best on cross-validated PR-AUC (ties broken by F1):", BEST)

In [ ]:
best_pipe = clone(models[BEST])

# Threshold tuned for F1 on out-of-fold TRAINING predictions only
oof = cross_val_predict(clone(best_pipe), Xtr, ytr, groups=gtr, cv=cv, method="predict_proba")[:, 1]
grid = np.linspace(0.1, 0.9, 81)
THR = grid[int(np.argmax([f1_score(ytr, oof >= t) for t in grid]))]
print("Tuned threshold: %.2f" % THR)

def metrics(y_true, proba, thr=0.5):
    pred = (proba >= thr).astype(int)
    return {"accuracy": accuracy_score(y_true, pred),
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred), "f1": f1_score(y_true, pred),
            "roc_auc": roc_auc_score(y_true, proba), "pr_auc": average_precision_score(y_true, proba)}

fitted, test_rows = {}, []
for name, pipe in models.items():
    p = clone(pipe).fit(Xtr, ytr); fitted[name] = p
    test_rows.append({"model": name, **metrics(yte, p.predict_proba(Xte)[:, 1])})
test_table = pd.DataFrame(test_rows).round(3)
display(test_table); test_table.to_csv(OUT_DIR / "stage1_test_results.csv", index=False)

p_te = fitted[BEST].predict_proba(Xte)[:, 1]
print(BEST, "at threshold 0.5   :", {k: round(v, 3) for k, v in metrics(yte, p_te).items()})
print(BEST, "at tuned threshold:", {k: round(v, 3) for k, v in metrics(yte, p_te, THR).items()})

ConfusionMatrixDisplay(confusion_matrix(yte, p_te >= THR),
                       display_labels=["Outside LF", "In LF"]).plot(cmap="Blues")
plt.title(f"Stage 1 - {BEST} (threshold {THR:.2f})")
plt.savefig(OUT_DIR / "fig_stage1_confusion.png", dpi=150); plt.show()

In [ ]:
# Odds ratios from logistic regression (numeric features are per 1 SD)
lr = clone(models["Logistic regression"]).fit(Xtr, ytr)
names = [n.replace("cat__", "").replace("num__", "") for n in lr.named_steps["pre"].get_feature_names_out()]
coef = pd.Series(lr.named_steps["model"].coef_[0], index=names)
odds = pd.DataFrame({"coef": coef, "odds_ratio": np.exp(coef)}).round(3).sort_values("coef")
print("Strongest positive associations:"); display(odds.tail(12).iloc[::-1])
print("Strongest negative associations:"); display(odds.head(12))
odds.to_csv(OUT_DIR / "stage1_odds_ratios.csv")

# Permutation importance for the best model (drop in ROC-AUC when a feature is shuffled)
samp = te.sample(min(15000, len(te)), random_state=SEED)
pi = permutation_importance(fitted[BEST], samp[FEATURES], samp["lfp"], scoring="roc_auc",
                            n_repeats=3 if FAST else 5, random_state=SEED)
imp = pd.Series(pi.importances_mean, index=FEATURES).sort_values()
imp.plot.barh(figsize=(7, 6), color="#1F3A5F")
plt.xlabel("Drop in ROC-AUC"); plt.title(f"Stage 1 permutation importance - {BEST}")
plt.tight_layout(); plt.savefig(OUT_DIR / "fig_stage1_importance.png", dpi=150); plt.show()
imp.sort_values(ascending=False).round(4).to_csv(OUT_DIR / "stage1_importance.csv")

In [ ]:
rows = []
for col, min_n in [("sector", 100), ("social_group", 100), ("religion", 100), ("state", 300)]:
    labels = lab(te[col], col).reset_index(drop=True)
    for g in sorted(labels.unique()):
        m = (labels == g).to_numpy()
        if m.sum() < min_n or yte.to_numpy()[m].min() == yte.to_numpy()[m].max():
            continue
        d = metrics(yte.to_numpy()[m], p_te[m], THR)
        rows.append({"variable": col, "group": g, "n": int(m.sum()),
                     "actual_rate": yte.to_numpy()[m].mean(), "predicted_rate": (p_te[m] >= THR).mean(),
                     "recall": d["recall"], "precision": d["precision"], "f1": d["f1"], "roc_auc": d["roc_auc"]})
fair = pd.DataFrame(rows).round(3)
display(fair[fair["variable"] != "state"])
print("States (largest gaps in F1 first):")
display(fair[fair["variable"] == "state"].sort_values("f1").head(10))
fair.to_csv(OUT_DIR / "stage1_subgroups.csv", index=False)